# 07 · Análisis de viabilidad

En `df_final` cada fase tiene su máquina dominante y la familia (grupo) de esa máquina. Con eso se puede montar un
cruce fase × máquina dentro de cada familia: ahí es donde vive la decisión de asignación.

La elección real no es "esta fase fue a esta máquina" (eso siempre pasa). La elección existe cuando fases del mismo
tipo pudieron ir a máquinas distintas de la misma familia. Por eso el análisis interesante es dentro de cada familia:
¿las fases de una familia se reparten entre sus varias máquinas, o cada tipo de fase va siempre a la misma?

- **Entrada:** `data/processed/df_final.parquet` y los embudos `results/tables/embudo_01.json` … `embudo_05.json`
- **Salida:** `results/tables/embudo_depuracion.csv` (tablas del embudo para la memoria)

In [7]:
import sys
from pathlib import Path

# Raíz del repositorio (la carpeta que contiene src/), para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.embudo import cargar_embudo, tabla_embudo
from src.rutas import PROCESSED, TABLES

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Embudo de depuración

Resumen de lo que queda tras cada paso de la preparación (notebooks 01 a 05).

Las fases siguen dos ramas que se unen en el notebook 05: las fases del ERP, que aportan el tiempo teórico, y las
fases del MES, construidas a partir de las imputaciones, que aportan el tiempo real. Son conjuntos distintos, así que
cada rama tiene su propia tabla. La unión se queda con las fases que están en las dos ramas.

In [8]:
e01, e02, e03, e04, e05 = (cargar_embudo(n) for n in ["01", "02", "03", "04", "05"])

bloques = {
    "Máquinas y familias": e02,
    "Imputaciones": {**e01, "Imputaciones en máquinas productivas": e03["Imputaciones en máquinas productivas"]},
    "Órdenes de fabricación": {**e04, "OF finales": e05["OF finales"]},
    "Fases · rama ERP": {k: e05[k] for k in ["Fases iniciales", "Fases sin externas", "Fases en OF válidas"]},
    "Fases · rama MES": {k: v for k, v in e03.items() if k.startswith("Fases")},
    "Fases · unión ERP–MES": {k: e05[k] for k in ["Fases con imputaciones validadas", "Fases con tiempos teóricos",
                                                  "Fases de trabajos válidos"]},
}

for nombre, embudo in bloques.items():
    print(f"\n{nombre}")
    display(tabla_embudo(embudo, "Número"))

filas = [{"bloque": b, "paso": p, "numero": n} for b, e in bloques.items() for p, n in e.items()]
pd.DataFrame(filas).to_csv(TABLES / "embudo_depuracion.csv", index=False)


Máquinas y familias


,Número
Máquinas iniciales,120
Máquinas activas en MES,119
Máquinas productivas,76
Máquinas con suficientes imputaciones,38
Máquinas en familias válidas,32
Familias iniciales,57
Familias finales,9



Imputaciones


,Número
Imputaciones iniciales,196576
Imputaciones dentro del intervalo temporal,140485
Imputaciones en máquinas productivas,59570



Órdenes de fabricación


...

,Número
Fases iniciales,116055
Fases sin externas,107542
Fases en OF válidas,69986



Fases · rama MES


,Número
Fases en máquinas productivas,19149
Fases válidas (dominante >80%),18548
Fases sin imputaciones mal cerradas,17294



Fases · unión ERP–MES


,Número
Fases con imputaciones validadas,16294
Fases con tiempos teóricos,14924
Fases de trabajos válidos,14689


## 2. Familias, máquinas y fases

In [9]:
df_final = pd.read_parquet(PROCESSED / "df_final.parquet")

resumen_familia = df_final.groupby("NombreGrupo").agg(
    n_fases=("SmFaseUid", "nunique"),
    n_maquinas=("MaquinaDominante", "nunique"),
).sort_values("n_fases", ascending=False)

print(f"Familias de máquinas: {len(resumen_familia)} | fases: {len(df_final)}")
resumen_familia

Familias de máquinas: 9 | fases: 14689


,n_fases,n_maquinas
NombreGrupo,,
Torns Puma (Taller COURES),4088,2
Màquines de Taladrat Profund,3602,2
Centres de Mecanitzat Vertical COURES,2633,4
Torns Control Numèric,1609,4
Centres de Mecanitzat Horitzontal COURES,1236,3
Centre de Mecanitzat 5 Eixos petites,547,4
Centres de Mecanitzat Vertical ALUMINIS,489,3
Premses Conformats,314,2
Màquines Secció Flexibles,171,4


## 3. Reparto del trabajo dentro de cada familia

Para cada familia se calcula qué porcentaje de sus fases se ejecuta en la máquina más utilizada (máquina principal),
cuántas máquinas se usan realmente y cuántos tipos de fase distintos hay.

In [10]:
filas = []
for familia, sub in df_final.groupby("NombreGrupo"):
    reparto = sub["MaquinaDominante"].value_counts(normalize=True)
    filas.append({"familia": familia,
                  "fases": len(sub),
                  "% máquina principal": round(reparto.iloc[0] * 100),
                  "máquinas usadas": int((reparto > 0.001).sum()),
                  "tipos de fase": sub["Description"].nunique()})

reparto_familias = pd.DataFrame(filas).sort_values("% máquina principal").set_index("familia")
reparto_familias

,fases,% máquina principal,máquinas usadas,tipos de fase
familia,,,,
Torns Control Numèric,1609,34,4,65
Centres de Mecanitzat Vertical COURES,2633,49,4,42
Premses Conformats,314,56,2,3
Centres de Mecanitzat Vertical ALUMINIS,489,65,3,10
Centres de Mecanitzat Horitzontal COURES,1236,73,3,33
Torns Puma (Taller COURES),4088,79,2,26
Centre de Mecanitzat 5 Eixos petites,547,80,4,36
Màquines de Taladrat Profund,3602,89,2,6
Màquines Secció Flexibles,171,98,4,2


Detalle de las fases de cada máquina dentro de su familia:

In [11]:
(df_final.groupby(["NombreGrupo", "MaquinaDominante"]).size()
 .rename("fases").reset_index()
 .sort_values(["NombreGrupo", "fases"], ascending=[True, False])
 .set_index(["NombreGrupo", "MaquinaDominante"]))

fases
NombreGrupo                              MaquinaDominante       
Centre de Mecanitzat 5 Eixos petites     CM12                439
                                         CM15                 86
                                         CM17                 21
                                         CM18                  1
Centres de Mecanitzat Horitzontal COURES CM05                908
                                         CM01                211
                                         CM07                117
Centres de Mecanitzat Vertical ALUMINIS  CM08                318
                                         CM10                154
                                         CM04                 17
Centres de Mecanitzat Vertical COURES    CM06               1286
                                         CM14                708
                                         CM16                600
                                         CM19                 39
Màquines Secció Flexibles                FL01                168
                                         CM11                  1
                                         CM13                  1
                                         MA02                  1
Màquines de Taladrat Profund             TP02               3188
                                         TP01                414
Premses Conformats                       PH04                175
                                         PH03                139
Torns Control Numèric                    TN08                554
                                         TN10                504
                                         TN05                423
                                         TN07                128
Torns Puma (Taller COURES)               TN09               3224
                                         TN04                864

## 4. Preparación y producción en el MES

En las imputaciones, a veces todo el tiempo se queda en preparación o en producción, sin separarlo, por la forma en
que los operarios registran el trabajo. Se comprueba cuántas fases tienen tiempo teórico en una parte pero tiempo
real 0 en esa misma parte:

In [12]:
prep_teorico_sin_real = ((df_final["PreparationTime"] > 0) & (df_final["MinutosPreparacion_efectivo"] == 0)).sum()
prod_teorico_sin_real = ((df_final["ExecutionTime"] > 0) & (df_final["MinutosProduccion_efectivo"] == 0)).sum()

print(f"Fases con tiempo teórico de preparación pero real de preparación = 0: {prep_teorico_sin_real}")
print(f"Fases con tiempo teórico de producción pero real de producción = 0: {prod_teorico_sin_real}")
print(f"Total de fases: {len(df_final)}")

Fases con tiempo teórico de preparación pero real de preparación = 0: 369
Fases con tiempo teórico de producción pero real de producción = 0: 12089
Total de fases: 14689


La mayoría de las fases tienen tiempo teórico de producción pero tiempo real de producción 0. La separación entre
preparación y producción en el MES no es fiable: el trabajo se imputa a una casilla u otra sin correspondencia con
cómo lo definió el ERP. Por eso el ratio se construye con el tiempo total (preparación + producción) en el
notebook 08.

## 5. Decisión de viabilidad

Tras el proceso de limpieza y depuración se dispone de un bloque sólido de 9 familias con elección real y volumen
suficiente:

- Torns Control Numèric — 34 % de concentración en la máquina principal, 4 máquinas utilizadas. El trabajo se reparte
  de forma equilibrada entre las cuatro, lo que valida la fusión de grupos realizada y constituye el ejemplo más claro
  de elección real. 1.641 fases.
- Centres de Mecanitzat Vertical COURES — 49 % de concentración, 4 máquinas utilizadas. Reparto saludable sobre
  2.636 fases.
- Premses Conformats — 56 % de concentración, 2 máquinas utilizadas. Reparto equilibrado entre las dos máquinas
  activas, con un volumen moderado (314 fases).
- Centres de Mecanitzat Vertical ALUMINIS — 65 % de concentración, 3 máquinas utilizadas. Reparto adecuado sobre
  507 fases.
- Centres de Mecanitzat Horitzontal COURES — 73 % de concentración, 3 máquinas utilizadas. Reparto aceptable sobre
  1.242 fases.

Estas cinco familias constituyen el núcleo sólido del análisis: presentan una decisión de asignación efectiva y un
reparto real del trabajo entre máquinas.

- Torns Puma (Taller COURES) — 79 % de concentración, 2 máquinas utilizadas. Aunque la máquina principal concentra la
  mayor parte del trabajo, el 21 % restante sobre 4.088 fases representa un volumen relevante en la segunda máquina.
- Centre de Mecanitzat 5 Eixos petites — 80 % de concentración, 4 máquinas utilizadas. La máquina principal domina,
  pero existe reparto entre las cuatro máquinas. 553 fases.
- Màquines de Taladrat Profund — 89 % de concentración, 2 máquinas utilizadas. Una máquina concentra casi todo el
  trabajo, aunque el 11 % restante sobre 3.604 fases sigue representando un volumen relevante en la segunda.

Estas tres familias se consideran frontera y se incluyen en el análisis.

- Màquines Secció Flexibles — 98 % de concentración y solo 174 fases. La máquina principal concentra casi la
  totalidad del trabajo sobre un volumen bajo, por lo que la elección resulta prácticamente inexistente. La familia
  se mantiene en el dataset y se menciona explícitamente como familia sin elección efectiva: en la evaluación
  (notebook 15) sus resultados se interpretan con cautela.

### Justificación

**Análisis de viabilidad: existencia de una decisión de asignación real**

Antes de abordar el modelado, resulta imprescindible verificar una premisa sobre la que se sostiene todo el trabajo:
que la asignación de una fase a una máquina constituye efectivamente una decisión con alternativas, y no un proceso
determinista en el que cada fase se ejecuta necesariamente en un único recurso posible. Si no existiera tal elección,
no habría idoneidad que evaluar ni asignación que mejorar, y el problema carecería de sentido.

Para comprobarlo, se ha analizado el conjunto de familias de máquina productivas resultantes del proceso de
depuración, cada una de las cuales agrupa máquinas funcionalmente equivalentes entre las que el taller puede repartir
el trabajo. Se ha exigido, como condición necesaria, que una familia disponga de al menos dos máquinas; las familias
constituidas por una única máquina se excluyeron de este análisis por no ofrecer alternativa alguna de asignación.
Sobre las familias restantes, se ha examinado no solo cuántas máquinas podrían ejecutar sus fases, sino en qué medida
el trabajo se reparte realmente entre ellas, calculando la proporción de fases que concentra la máquina más
utilizada de cada familia.

Los resultados confirman la existencia de una decisión de asignación efectiva. En la mayoría de las familias
analizadas, la máquina principal concentra entre el 34 % y el 73 % de las fases, evidenciando que el trabajo se
distribuye de forma genuina entre varios recursos y que, por tanto, existe una elección real en la asignación. Casos
como el de los tornos de control numérico (34 % en la máquina principal, con las fases repartidas entre cuatro
máquinas) o los centros de mecanizado vertical de cobre (49 %) constituyen la evidencia más clara de que la
asignación no es determinista. Se observaron asimismo familias con una concentración elevada en una única máquina
(superior al 85 %), que, si bien disponen de alternativas, presentan una elección efectiva más limitada; estas se
tratan con cautela en el análisis posterior.

En conjunto, este análisis valida empíricamente la premisa central del trabajo: existe un problema de asignación
fase-máquina con capacidad de elección real, sobre un volumen de datos suficiente, que un modelo de idoneidad puede
aprender a caracterizar y mejorar.

### Conclusión

De las familias productivas con varias máquinas, la mayoría presentan un reparto efectivo del trabajo entre sus
máquinas, lo que confirma que la asignación fase-máquina constituye una decisión real y no determinista en el taller.

A partir de 196.576 imputaciones en crudo (140.485 dentro del intervalo de estudio), el proceso de depuración da una
población de 14.689 fases analizables, cada una con su máquina, su familia y sus tiempos real y teórico. Es una base
amplia y suficiente para el modelado posterior.